# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Roselyn-Koech/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

## Two paper findings + my methodology questions

### Finding 1: Content Lifecycle — Growing vs Declining

The FlyRank research paper reports a distinction between content that is growing and content that is declining, using observed performance trends in the study dataset.

**Methodology question:**

How exactly is the growing versus declining label constructed, and what time window is used to determine the direction of the trend? I would want to confirm that the information used to create the label comes from the defined outcome period and is not also included among the features used to make the prediction. This matters because using outcome-period information as a feature could make the measured model performance look stronger than it would be in a real decision setting.

I would also want to understand whether the same content or client can appear across both the analysis and validation periods, since related observations could make the validation result less independent.


### Finding 2: The CTR Cliff

The FlyRank research paper reports a relationship between click-through rate (CTR) and content performance, describing a point at which lower CTR is associated with weaker performance.

**Methodology question:**

How was the relationship between CTR and performance measured, and how was the relevant comparison or threshold selected? I would want to know whether this is an observed relationship in the dataset or whether the analysis supports a causal interpretation.

I would also check whether CTR was measured before the outcome period being analyzed. If CTR and the outcome are measured over overlapping periods, the relationship may be useful for describing the data but should be treated carefully when using CTR as a predictive feature.



### Why I chose these questions

I am not treating these questions as evidence that the research findings are incorrect. The purpose is to identify the assumptions and validation choices that matter when interpreting the findings.

The same questions are relevant to my own model: where the label comes from, whether the features would genuinely be available at prediction time, and whether the validation design represents the way the model would be used in practice.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [6]:
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier


RANDOM_STATE = 42

DATA_URL = (
    "https://raw.githubusercontent.com/"
    "Roselyn-Koech/flyrank-ml-internship/"
    "main/data/raw/content_refresh_anonymized.csv"
)

# ------------------------------------------------------------
# 1. Load and prepare the same data used in Week 5
# ------------------------------------------------------------

local_candidates = [
    Path("data/raw/content_refresh_anonymized.csv"),
    Path("/content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv")
]

DATA_PATH = next((p for p in local_candidates if p.exists()), None)

df = (
    pd.read_csv(DATA_PATH)
    if DATA_PATH
    else pd.read_csv(DATA_URL)
)

# Match Week 5 preparation rules
df = df[
    (df["impressions_90d"] > 0)
    & (df["content_age_days"] >= 90)
].copy()

df = df.drop_duplicates("content_id").reset_index(drop=True)

# Target definition
df["is_declining_label"] = (
    df["trend_direction"]
    .astype(str)
    .str.lower()
    .eq("down")
    .astype(int)
)

# Numeric preparation
numeric_raw = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "users_90d",
    "engaged_sessions_90d",
    "ai_sessions_90d",
    "scroll_events_90d",
    "days_with_impressions",
    "days_with_sessions",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct",
]

for c in numeric_raw:
    df[c] = (
        pd.to_numeric(df[c], errors="coerce")
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0)
    )

# Log transforms used in Week 5
df["log_impressions_90d"] = np.log1p(df["impressions_90d"])
df["log_clicks_90d"] = np.log1p(df["clicks_90d"])
df["log_sessions_90d"] = np.log1p(df["sessions_90d"])
df["log_ai_sessions_90d"] = np.log1p(df["ai_sessions_90d"])

numeric_features = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "log_impressions_90d",
    "log_clicks_90d",
    "log_sessions_90d",
    "log_ai_sessions_90d",
    "days_with_impressions",
    "days_with_sessions",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct",
]

categorical_features = [
    "competition_level",
    "content_type",
    "main_intent",
    "age_tier",
    "freshness_tier",
    "word_count_tier",
    "impression_tier",
    "position_tier",
]

for c in categorical_features:
    df[c] = df[c].fillna("unknown").astype(str)


# ------------------------------------------------------------
# 2. Honest grouped split by client
# ------------------------------------------------------------

clients = df["client_id"].fillna("unknown").astype(str)

unique_clients = clients.drop_duplicates().to_numpy()

rng = np.random.default_rng(RANDOM_STATE)

shuffled_clients = rng.permutation(unique_clients)

test_client_count = max(
    1,
    int(round(len(shuffled_clients) * 0.20))
)

test_clients = set(
    shuffled_clients[:test_client_count]
)

test_mask = clients.isin(test_clients).to_numpy()

train_idx = np.where(~test_mask)[0]
test_idx = np.where(test_mask)[0]

X = df[numeric_features + categorical_features]
y = df["is_declining_label"]

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

print("HONEST VALIDATION DESIGN")
print("=" * 60)
print("Split type: client-grouped holdout")
print(f"Total rows: {len(df):,}")
print(f"Training rows: {len(train_idx):,}")
print(f"Test rows: {len(test_idx):,}")
print(f"Total clients: {len(unique_clients)}")
print(f"Held-out clients: {len(test_clients)}")
print(f"Training clients: {len(unique_clients) - len(test_clients)}")
print()
print("Held-out clients:")
print(sorted(test_clients))
print()
print("Training label rate:", round(y_train.mean(), 3))
print("Test label rate:", round(y_test.mean(), 3))


# ------------------------------------------------------------
# 3. Confirm that no client appears in both sets
# ------------------------------------------------------------

train_clients = set(clients.iloc[train_idx])
test_clients_check = set(clients.iloc[test_idx])

overlap = train_clients.intersection(test_clients_check)

print()
print("CLIENT OVERLAP CHECK")
print("=" * 60)
print("Clients appearing in both train and test:", len(overlap))

assert len(overlap) == 0, (
    "Validation error: client overlap detected."
)

print("PASS: No client appears in both training and test sets.")


# ------------------------------------------------------------
# 4. Build the same model family used in Week 5
# ------------------------------------------------------------

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numeric_features
        ),
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            categorical_features
        ),
    ]
)

models = {
    "logistic_regression": LogisticRegression(
        max_iter=1000,
        random_state=RANDOM_STATE
    ),

    "decision_tree": DecisionTreeClassifier(
        max_depth=5,
        random_state=RANDOM_STATE
    ),

    "random_forest": RandomForestClassifier(
        n_estimators=200,
        max_depth=8,
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),
}


# ------------------------------------------------------------
# 5. Precision@K helper
# ------------------------------------------------------------

def precision_at_k(y_true, scores, k):
    y_true = np.asarray(y_true)
    scores = np.asarray(scores)

    k = min(k, len(y_true))

    top_idx = np.argsort(scores)[::-1][:k]

    return y_true[top_idx].mean()


# ------------------------------------------------------------
# 6. Train and evaluate models on unseen clients
# ------------------------------------------------------------

results = {}

for name, model in models.items():

    pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", model)
        ]
    )

    pipeline.fit(X_train, y_train)

    probabilities = pipeline.predict_proba(X_test)[:, 1]

    predictions = (
        probabilities >= 0.5
    ).astype(int)

    results[name] = {
        "ROC AUC": roc_auc_score(
            y_test,
            probabilities
        ),

        "Average Precision": average_precision_score(
            y_test,
            probabilities
        ),

        "Precision@20": precision_at_k(
            y_test,
            probabilities,
            20
        ),

        "Precision@50": precision_at_k(
            y_test,
            probabilities,
            50
        ),

        "Precision": precision_score(
            y_test,
            predictions,
            zero_division=0
        ),

        "Recall": recall_score(
            y_test,
            predictions,
            zero_division=0
        ),

        "F1": f1_score(
            y_test,
            predictions,
            zero_division=0
        ),
    }


# ------------------------------------------------------------
# 7. Rebuild the Week-4 baseline on the same test rows
# ------------------------------------------------------------

test_df = df.iloc[test_idx].copy()

# Normalize the components used by the baseline
ctr_rank = test_df["ctr"].rank(pct=True)

position_rank = (
    1 - test_df["avg_position"].rank(pct=True)
)

impression_rank = (
    test_df["impressions_90d"].rank(pct=True)
)

baseline_score = (
    0.40 * ctr_rank
    + 0.35 * position_rank
    + 0.25 * impression_rank
)

results["baseline_rules"] = {
    "ROC AUC": roc_auc_score(
        y_test,
        baseline_score
    ),

    "Average Precision": average_precision_score(
        y_test,
        baseline_score
    ),

    "Precision@20": precision_at_k(
        y_test,
        baseline_score,
        20
    ),

    "Precision@50": precision_at_k(
        y_test,
        baseline_score,
        50
    ),

    "Precision": np.nan,
    "Recall": np.nan,
    "F1": np.nan,
}


# ------------------------------------------------------------
# 8. Show the honest validation results
# ------------------------------------------------------------

honest_results = (
    pd.DataFrame(results)
    .T
    .sort_values(
        "Precision@50",
        ascending=False
    )
)

print()
print("HONEST CLIENT-GROUPED VALIDATION RESULTS")
print("=" * 60)

display(
    honest_results.round(3)
)


# ------------------------------------------------------------
# 9. Compare against the Week-5 reported results
# ------------------------------------------------------------

week5_results = pd.DataFrame(
    {
        "ROC AUC": {
            "baseline_rules": 0.627,
            "logistic_regression": 0.704,
            "decision_tree": 0.742,
            "random_forest": 0.750,
        },

        "Average Precision": {
            "baseline_rules": 0.468,
            "logistic_regression": 0.525,
            "decision_tree": 0.575,
            "random_forest": 0.618,
        },

        "Precision@20": {
            "baseline_rules": 0.15,
            "logistic_regression": 0.35,
            "decision_tree": 0.75,
            "random_forest": 0.65,
        },

        "Precision@50": {
            "baseline_rules": 0.24,
            "logistic_regression": 0.40,
            "decision_tree": 0.78,
            "random_forest": 0.74,
        },
    }
)

print()
print("WEEK-5 REPORTED RESULTS")
print("=" * 60)

display(
    week5_results.round(3)
)


comparison = honest_results[
    [
        "ROC AUC",
        "Average Precision",
        "Precision@20",
        "Precision@50"
    ]
].join(
    week5_results,
    lsuffix="_honest",
    rsuffix="_week5"
)

print()
print("BEFORE / AFTER COMPARISON")
print("=" * 60)

display(
    comparison.round(3)
)

HONEST VALIDATION DESIGN
Split type: client-grouped holdout
Total rows: 30,000
Training rows: 27,675
Test rows: 2,325
Total clients: 32
Held-out clients: 6
Training clients: 26

Held-out clients:
['client_0b918943df', 'client_1a6562590e', 'client_4fc82b26ae', 'client_98a3ab7c34', 'client_d4735e3a26', 'client_f74efabef1']

Training label rate: 0.555
Test label rate: 0.391

CLIENT OVERLAP CHECK
Clients appearing in both train and test: 0
PASS: No client appears in both training and test sets.

HONEST CLIENT-GROUPED VALIDATION RESULTS


,ROC AUC,Average Precision,Precision@20,Precision@50,Precision,Recall,F1
random_forest,0.745,0.611,0.80,0.72,0.522,0.836,0.642
decision_tree,0.741,0.574,0.50,0.58,0.568,0.724,0.637
baseline_rules,0.480,0.389,0.40,0.40,NaN,NaN,NaN
logistic_regression,0.703,0.521,0.35,0.34,0.569,0.651,0.607



WEEK-5 REPORTED RESULTS


,ROC AUC,Average Precision,Precision@20,Precision@50
baseline_rules,0.627,0.468,0.15,0.24
logistic_regression,0.704,0.525,0.35,0.40
decision_tree,0.742,0.575,0.75,0.78
random_forest,0.750,0.618,0.65,0.74



BEFORE / AFTER COMPARISON


,ROC AUC_honest,Average Precision_honest,Precision@20_honest,Precision@50_honest,ROC AUC_week5,Average Precision_week5,Precision@20_week5,Precision@50_week5
random_forest,0.745,0.611,0.80,0.72,0.750,0.618,0.65,0.74
decision_tree,0.741,0.574,0.50,0.58,0.742,0.575,0.75,0.78
baseline_rules,0.480,0.389,0.40,0.40,0.627,0.468,0.15,0.24
logistic_regression,0.703,0.521,0.35,0.34,0.704,0.525,0.35,0.40


### Interpretation of the honest validation

The Week-5 model was already evaluated using a client holdout, so this audit does not treat the original evaluation as a random row split. Instead, I independently reconstructed the client-grouped validation design and verified that no client appeared in both the training and test sets.

The honest split contained 27,675 training rows from 26 clients and 2,325 test rows from 6 completely held-out clients. The client overlap check returned zero.

The Random Forest remained the strongest model under this evaluation. Its ROC AUC changed from 0.750 in the Week-5 reported results to 0.745 under the independently reconstructed client-grouped split. Average Precision changed from 0.618 to 0.611. Precision@50 changed from 0.74 to 0.72.

Precision@20 increased from 0.65 to 0.80 in the honest evaluation. Because Precision@20 is based on only the top 20 predictions, I would not treat this increase alone as evidence of general improvement.

The held-out test clients also had a lower declining-label rate than the training clients (39.1% compared with 55.5%). This provides a useful distribution shift check because the test clients were not identical to the training population.

Overall, the Random Forest's ROC AUC and Average Precision were close to the Week-5 results under the independently reconstructed grouped validation. This supports the narrower claim that the measured performance was reasonably stable under this client-grouped evaluation.

I would not claim that the model will generalize to every future client. The evaluation covers the available anonymized clients and provides directional evidence for decision support rather than proof of universal performance.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [7]:
# ============================================================
# SECTION 3: LEAKAGE AUDIT
# ============================================================

print("LEAKAGE AUDIT")
print("=" * 60)

# ------------------------------------------------------------
# 1. Identify the target definition
# ------------------------------------------------------------

print("Target:")
print("is_declining_label = (trend_direction == 'down')")
print()

# ------------------------------------------------------------
# 2. Columns that directly define or describe the target
# ------------------------------------------------------------

direct_leakage = [
    "trend_direction",
    "trend_pct"
]

print("Direct target-related columns:")
for col in direct_leakage:
    print(f"  EXCLUDED: {col}")

# ------------------------------------------------------------
# 3. Identifier columns
# ------------------------------------------------------------

identifier_columns = [
    "client_id",
    "content_id"
]

print()
print("Identifier columns:")
for col in identifier_columns:
    print(f"  EXCLUDED FROM FEATURES: {col}")

# ------------------------------------------------------------
# 4. Verify the actual feature set
# ------------------------------------------------------------

feature_columns = numeric_features + categorical_features

print()
print("FEATURE LEAKAGE CHECK")
print("=" * 60)

print(f"Number of model features: {len(feature_columns)}")

found_leakage = [
    col for col in direct_leakage
    if col in feature_columns
]

found_identifiers = [
    col for col in identifier_columns
    if col in feature_columns
]

print()
print("Target-defining columns included as features:", found_leakage)
print("Identifier columns included as features:", found_identifiers)

assert len(found_leakage) == 0, (
    "Leakage detected: target-defining column included."
)

assert len(found_identifiers) == 0, (
    "Identifier leakage detected."
)

print()
print("PASS: No target-defining or identifier columns are used as model features.")

# ------------------------------------------------------------
# 5. Check that every feature exists in the source dataframe
# ------------------------------------------------------------

missing_features = [
    col for col in feature_columns
    if col not in df.columns
]

print()
print("Missing feature columns:", missing_features)

assert len(missing_features) == 0

# ------------------------------------------------------------
# 6. Check whether target-related columns are correlated
#    with the target because they define it
# ------------------------------------------------------------

print()
print("TARGET CONSTRUCTION CHECK")
print("=" * 60)

print(
    "The label is constructed directly from trend_direction."
)

print(
    "Therefore trend_direction must never be used as a predictor."
)

print(
    "trend_pct is also excluded because it describes the same "
    "declining trend information and could expose outcome-period "
    "information."
)

# ------------------------------------------------------------
# 7. Summary table
# ------------------------------------------------------------

leakage_audit = pd.DataFrame({
    "Column": [
        "trend_direction",
        "trend_pct",
        "client_id",
        "content_id"
    ],

    "Risk": [
        "Direct target definition",
        "Outcome/trend information",
        "Identifier / grouping variable",
        "Identifier"
    ],

    "Decision": [
        "EXCLUDE",
        "EXCLUDE",
        "EXCLUDE",
        "EXCLUDE"
    ],

    "Reason": [
        "Defines is_declining_label",
        "May expose outcome-period trend information",
        "Used only to create grouped validation",
        "Identifies individual content"
    ]
})

display(leakage_audit)

print()
print("FINAL LEAKAGE AUDIT RESULT: PASS")

LEAKAGE AUDIT
Target:
is_declining_label = (trend_direction == 'down')

Direct target-related columns:
  EXCLUDED: trend_direction
  EXCLUDED: trend_pct

Identifier columns:
  EXCLUDED FROM FEATURES: client_id
  EXCLUDED FROM FEATURES: content_id

FEATURE LEAKAGE CHECK
Number of model features: 26

Target-defining columns included as features: []
Identifier columns included as features: []

PASS: No target-defining or identifier columns are used as model features.

Missing feature columns: []

TARGET CONSTRUCTION CHECK
The label is constructed directly from trend_direction.
Therefore trend_direction must never be used as a predictor.
trend_pct is also excluded because it describes the same declining trend information and could expose outcome-period information.


,Column,Risk,Decision,Reason
0,trend_direction,Direct target definition,EXCLUDE,Defines is_declining_label
1,trend_pct,Outcome/trend information,EXCLUDE,May expose outcome-period trend information
2,client_id,Identifier / grouping variable,EXCLUDE,Used only to create grouped validation
3,content_id,Identifier,EXCLUDE,Identifies individual content



FINAL LEAKAGE AUDIT RESULT: PASS


## Real failure examples

The model was evaluated on clients that were completely held out from training. I inspected individual false positives and false negatives from this test set to understand where the model made mistakes.

A false positive is a page that the model ranked as likely to be declining but whose observed label was not declining.

A false negative is a page whose observed label was declining but which the model did not rank highly enough.

The purpose of this analysis is not to identify a single cause for each error, but to understand patterns in the cases where the model's predictions did not match the observed label.

In [8]:
# ============================================================
# SECTION 4A: REAL FAILURE EXAMPLES
# ============================================================

# Refit the Random Forest so we have its predictions available
rf_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            RandomForestClassifier(
                n_estimators=200,
                max_depth=8,
                random_state=RANDOM_STATE,
                n_jobs=-1
            )
        )
    ]
)

rf_pipeline.fit(X_train, y_train)

test_probabilities = rf_pipeline.predict_proba(X_test)[:, 1]

failure_df = df.iloc[test_idx].copy()

failure_df["actual_label"] = y_test.to_numpy()
failure_df["predicted_probability"] = test_probabilities
failure_df["predicted_label"] = (
    test_probabilities >= 0.5
).astype(int)

# ------------------------------------------------------------
# False positives
# ------------------------------------------------------------

false_positives = failure_df[
    (failure_df["predicted_label"] == 1)
    & (failure_df["actual_label"] == 0)
].copy()

# Highest-confidence false positives
false_positives = false_positives.sort_values(
    "predicted_probability",
    ascending=False
)

print("FALSE POSITIVES")
print("=" * 60)

display(
    false_positives[
        [
            "client_id",
            "content_id",
            "predicted_probability",
            "actual_label",
            "ctr",
            "avg_position",
            "impressions_90d",
            "content_age_days",
            "days_since_last_update"
        ]
    ].head(10)
)


# ------------------------------------------------------------
# False negatives
# ------------------------------------------------------------

false_negatives = failure_df[
    (failure_df["predicted_label"] == 0)
    & (failure_df["actual_label"] == 1)
].copy()

# Highest-risk false negatives
false_negatives = false_negatives.sort_values(
    "predicted_probability",
    ascending=True
)

print()
print("FALSE NEGATIVES")
print("=" * 60)

display(
    false_negatives[
        [
            "client_id",
            "content_id",
            "predicted_probability",
            "actual_label",
            "ctr",
            "avg_position",
            "impressions_90d",
            "content_age_days",
            "days_since_last_update"
        ]
    ].head(10)
)


# ------------------------------------------------------------
# Failure counts
# ------------------------------------------------------------

fp_count = len(false_positives)
fn_count = len(false_negatives)

print()
print("FAILURE SUMMARY")
print("=" * 60)
print(f"False positives: {fp_count:,}")
print(f"False negatives: {fn_count:,}")

FALSE POSITIVES


,client_id,content_id,predicted_probability,actual_label,ctr,avg_position,impressions_90d,content_age_days,days_since_last_update
25913,client_f74efabef1,content_331182ca4cae,0.757697,0,0.00,35.9,3026,134,20
23750,client_f74efabef1,content_e55b8ab078b0,0.752463,0,0.00,21.8,369,112,20
10155,client_f74efabef1,content_643f585dc7f7,0.749137,0,0.39,25.1,761,104,20
21530,client_f74efabef1,content_b15a8dbdf66f,0.746002,0,0.18,22.4,1647,144,20
23559,client_f74efabef1,content_00603b0349b4,0.744862,0,0.09,25.6,1076,125,20
4249,client_f74efabef1,content_db1cd41b4b4f,0.743643,0,0.00,12.9,1482,105,105
15615,client_f74efabef1,content_daa53fb38efa,0.743371,0,0.00,14.3,2876,134,20
23250,client_f74efabef1,content_d2dffcc697a4,0.742002,0,0.20,14.1,5091,144,20
5966,client_f74efabef1,content_f5013794ba57,0.740064,0,0.00,15.7,881,175,20
28365,client_f74efabef1,content_24f8c25672ea,0.739692,0,0.00,13.1,950,112,20



FALSE NEGATIVES


,client_id,content_id,predicted_probability,actual_label,ctr,avg_position,impressions_90d,content_age_days,days_since_last_update
5770,client_98a3ab7c34,content_28b4223f4e5f,0.073246,1,0.00,0.0,1,91,1
3879,client_d4735e3a26,content_34b14c00f80c,0.112162,1,0.00,0.0,3,308,20
27177,client_f74efabef1,content_79ac977c6e0b,0.165234,1,0.00,0.7,3,104,8
25838,client_98a3ab7c34,content_cbc3b52a2ac1,0.198067,1,0.00,3.0,2,125,1
22991,client_d4735e3a26,content_472ce7ae14c0,0.217843,1,33.33,0.3,3,300,20
24200,client_0b918943df,content_2cfc7a1fc728,0.227630,1,0.00,5.0,3,321,20
12864,client_d4735e3a26,content_f1ef151d5e36,0.229917,1,0.00,2.0,3,294,20
19156,client_d4735e3a26,content_c62510afc57d,0.231377,1,0.00,2.0,1,489,20
25755,client_d4735e3a26,content_a8cee66e4788,0.241673,1,100.00,2.0,1,489,20
5608,client_d4735e3a26,content_a55d958ec725,0.241774,1,0.00,2.7,3,290,20



FAILURE SUMMARY
False positives: 697
False negatives: 149


### Failure analysis interpretation

The Random Forest produced 697 false positives and 149 false negatives at the default probability threshold of 0.5 on the held-out test clients.

The false-positive examples show that the model sometimes assigns high declining probabilities to pages with weak observed performance signals, such as zero or very low CTR, weaker average position, and varying levels of impressions. For example, the highest-confidence false positive had a predicted probability of 0.758, zero CTR, an average position of 35.9, and 3,026 impressions, but its observed label was 0.

The false-negative examples include several pages with very sparse performance signals. Several had zero CTR and very low impression counts, including examples with only 1 to 3 impressions. These cases were labeled as declining even though the model assigned relatively low probabilities.

These examples suggest that the model can struggle to distinguish some low-performance pages from pages that receive the declining label. However, these examples are descriptive rather than causal evidence. I therefore do not claim that sparse traffic causes model errors.

Because the project is intended to prioritize pages for review, ranking metrics such as Precision@20 and Precision@50 are more directly useful than treating the 0.5 classification threshold as a final business decision rule.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

## Claim rewrite

### Original claim

A strong version of my original claim could be interpreted as:

> "The Random Forest model accurately predicts which pages are declining and can identify the pages that should be refreshed."

This wording goes further than the evidence supports because the evaluation does not prove that the model will correctly identify every declining page or that refreshing a page based on the prediction will improve its performance.

### Evidence-supported claim

I would now state the result more carefully:

> "Under a client-grouped validation design, the Random Forest produced measured ROC AUC of 0.745 and Average Precision of 0.611 on held-out clients. Its Precision@50 was 0.72 in this evaluation. These results provide directional decision support for prioritizing content pages for review based on observed performance signals."

### Why I changed the claim

The revised claim uses "measured" and "directional decision support" because the evaluation provides evidence about predictive performance, not proof that the model will improve business outcomes.

The evaluation also covers a finite set of anonymized clients rather than every possible future client. Although the Random Forest results were reasonably close to the Week-5 reported results, this does not establish universal generalization.

I therefore avoid claiming that the model is universally accurate, that it causes better content performance, or that every page it identifies should be refreshed.

### Safe language used throughout this audit

I use:

- observed
- measured
- evaluated
- associated with
- directional
- decision support
- prioritization

I avoid unsupported language such as:

- proves
- guarantees
- causes
- always
- universally accurate
- will improve performance

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.